In [1]:
# https://learnopencv.com/fine-tuning-bert/

In [2]:
from pathlib import Path
from typing import Literal

import torch

from collections.abc import Callable

from datasets import Dataset, load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate
import glob
import json
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from pathlib import Path
from datetime import datetime

import sys

sys.path.insert(0, str(Path.cwd().parent))


from finetuning.commons import PipelineData, prepare_data, parse_pubtator, build_training_samples, samples_to_rels_like_df
from dataset_preparation.perturbations import BIORED_RELATION_TYPES, NO_RELATION_LABEL
from dataset_preparation.prepare_pure_biored import build_pure_biored_samples, compute_relation_distance_stats

/nix/store/2cp994c7ginqfxx7m6j0zzzdqkbavi4y-python3-3.13.13-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

USE_CACHE = True

DATASET_NAME: Literal["BioRed", "BioRedPerturbated"] = "BioRedPerturbated"

F1_AVERAGE: Literal["micro", "macro", "weighted"] = "micro"

OUT_DIR = f"relations-bert-{datetime.now():%Y-%m-%d-%H-%M-%S}_{DATASET_NAME}_{F1_AVERAGE}"

In [ ]:



# NOTE: these are the defaults might change according to avaibale VRAM
# -> BATCH SIZE and LR are halved if less than 8GB of VRAM is detected
BATCH_SIZE = 32
NUM_PROCS = 32
LR = 0.00005
EPOCHS = 10
# MODEL = 'NeuML/pubmedbert-base-embeddings'
MODEL = 'bioformers/bioformer-8L'
CACHE_DIR = Path("cache")

PUBTATOR_FILE_TRAIN = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Train.PubTator"
PUBTATOR_FILE_DEV = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Dev.PubTator"
PUBTATOR_FILE_TEST = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Test.PubTator"



def load_or_cache(split: str, prefix: str, build: Callable[[], Dataset], use_cache: bool = True) -> Dataset:
    """Load a Hugging Face ``Dataset`` from disk cache or build and persist it.

    Cached data is stored under ``CACHE_DIR / f"{prefix}_{split}"`` (default: ``cache/``).
    On a cache hit, ``load_from_disk`` is used and ``build`` is not called.
    On a miss, ``build()`` runs once, the result is saved with ``save_to_disk``, then returned.

    Args:
        split: Split identifier used in the cache directory name (e.g. ``"train"``, ``"validation"``).
        prefix: Stage prefix distinguishing pipeline steps (e.g. ``"raw"``, ``"tokenized"``).
        build: Zero-argument callable that produces the dataset when the cache is missing.

    Returns:
        The dataset for the given split, either loaded from cache or freshly built.

    Examples:
        Download a Hub split and cache it as ``cache/raw_train/``::

            train = load_or_cache(
                "train",
                "raw",
                lambda: load_dataset("ccdv/arxiv-classification", split="train"),
            )

        Tokenize an in-memory split and cache as ``cache/tokenized_train/``::

            tokenized_train = load_or_cache(
                "train",
                "tokenized",
                lambda: train.map(preprocess_function, batched=True, batch_size=32),
            )

    Note:
        Delete the matching folder under ``cache/`` to force a rebuild after changing
        ``build``, the source data, or preprocessing.
    """
    cache_path = CACHE_DIR / f"{prefix}_{split}"
    if cache_path.exists() and use_cache:
        print(f"Loading {prefix} {split} from cache/")
        return load_from_disk(cache_path)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset = build()
    dataset.save_to_disk(cache_path)
    return dataset


In [5]:
MODE: str = "cpu"
if torch.backends.mps.is_available():
    MODE = "mps"
elif torch.cuda.is_available():
    MODE = "cuda"
else:
    print("No GPU or MPS available - uising CPU")

print(f"Using {MODE} for training")


hardware_specific_args = {}

if MODE == "mps":
    hardware_specific_args["fp16"] = False
    hardware_specific_args["dataloader_num_workers"] = 0
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE
    hardware_specific_args["learning_rate"] = LR
    

elif MODE == "cuda":
    # Check total GPU VRAM 
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA device VRAM: {total_vram_gb:.2f} GB")

    # Default: assume >8GB VRAM
    batch_div = 1
    lr_div = 1

    # 2070super only has 8gigs of VRAM :')
    if total_vram_gb <= 8.5:
        print("Detected ~8GB of VRAM or less, reducing batch size and learning rate.")
        batch_div = 2
        lr_div = 2

    hardware_specific_args["fp16"] = True
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["learning_rate"] = LR / lr_div

    print(f"Using {hardware_specific_args['per_device_train_batch_size']} for training")
    print(f"Using {hardware_specific_args['per_device_eval_batch_size']} for evaluation")
    print(f"Using {hardware_specific_args['learning_rate']} for learning rate")



Using cuda for training
CUDA device VRAM: 7.57 GB
Detected ~8GB of VRAM or less, reducing batch size and learning rate.
Using 16 for training
Using 16 for evaluation
Using 2.5e-05 for learning rate


In [6]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_micro",
    greater_is_better=True,
    save_total_limit=EPOCHS,
    report_to="tensorboard",
    **hardware_specific_args,
)

In [7]:
RELATION_LABELS: list[str] = BIORED_RELATION_TYPES + [NO_RELATION_LABEL]
label2id: dict[str, int] = {name: idx for idx, name in enumerate(RELATION_LABELS)}
id2label: dict[int, str] = {idx: name for name, idx in label2id.items()}
MASK_TOKEN = "[MASK]"

def build_samples(
    pubtator_file: Path,
    dataset_name: Literal["BioRed", "BioRedPerturbated"] = DATASET_NAME,
    label_map: dict[str, int] = label2id,
    mask_token: str = MASK_TOKEN,
    no_relation_label: str = NO_RELATION_LABEL,
    parsed: tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame] | None = None,
    distance_stats: dict[str, float] | None = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """Parse a PubTator file and build the relation-classification samples DataFrame.

    Args:
        pubtator_file: Path to the PubTator file to parse.
        dataset_name: Which sample-building strategy to use. ``"BioRed"`` uses gold
            relations plus distance-matched NoRelation examples, ``"BioRedPerturbated"``
            uses the perturbated training samples.
        label_map: Mapping from relation label name to integer class id.
        mask_token: Token used as the relation placeholder in the prompt.
        no_relation_label: Label assigned to ``false_positive`` (unrelated) pairs.
        parsed: Optional pre-parsed ``(meta, anns, rels)`` tuple to avoid re-reading the file.
        distance_stats: Train-fit distance thresholds for NoRelation sampling on dev/test.
        verbose: Whether to print NoRelation sampling diagnostics.

    Returns:
        A DataFrame with ``prompt``, ``target_relation`` and ``label`` columns,
        restricted to ``gold`` and ``false_positive`` perturbations.
    """
    if parsed is not None:
        meta_df, anns_df, rels_df = parsed
    else:
        meta_df, anns_df, rels_df = parse_pubtator(pubtator_file)

    if dataset_name == "BioRedPerturbated":
        samples = build_training_samples(meta_df, anns_df, rels_df)

    elif dataset_name == "BioRed":
        # Original BioRED: gold relations + distance-matched NoRelation examples.
        samples = build_pure_biored_samples(
            meta_df,
            anns_df,
            rels_df,
            distance_stats=distance_stats,
            verbose=verbose,
        )

    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name!r}")

    samples = samples_to_rels_like_df(samples)

    # Gold relations (8 types) + unrelated entity pairs (NoRelation).
    samples = samples[samples["perturbation"].isin(["gold", "false_positive"])].copy()

    samples["prompt"] = samples.apply(
        lambda row: (
            f"Relation: {row['entity_a_text']} -> {mask_token} -> {row['entity_b_text']}\n"
            f"Context: {row['abstract']}"
        ),
        axis=1,
    )
    samples["target_relation"] = np.where(
        samples["perturbation"] == "false_positive",
        no_relation_label,
        samples["relation_type"],
    )
    samples["label"] = samples["target_relation"].map(label_map)
    return samples


# Official BioRED splits: Train for fitting, Dev for validation/checkpoint selection, Test held out.
train_parsed = parse_pubtator(PUBTATOR_FILE_TRAIN)
train_distance_stats = compute_relation_distance_stats(*train_parsed)

train_df = build_samples(
    PUBTATOR_FILE_TRAIN,
    parsed=train_parsed,
    distance_stats=train_distance_stats,
)
val_df = build_samples(
    PUBTATOR_FILE_DEV,
    distance_stats=train_distance_stats,
    verbose=False,
)
test_df = build_samples(
    PUBTATOR_FILE_TEST,
    distance_stats=train_distance_stats,
    verbose=False,
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
print(f"Classes: {len(RELATION_LABELS)}")
print(f"Train: {len(train_df)}, Dev: {len(val_df)}, Test: {len(test_df)}")
print(train_df["target_relation"].value_counts())

Related-entity distance statistics (characters):
  pairs  : 4178
  mean   : 73.8
  median : 37.0
  std    : 121.7
  min    : 0.0
  max    : 1599.0
{'pmid': '10491763', 'relation_type': 'Association', 'id_1': '3175', 'id_2': 'D003924', 'entity_a_text': 'hepatocyte nuclear factor (HNF)-6', 'entity_b_text': 'Type II (non-insulin-dependent) diabetes mellitus', 'perturbation': 'gold', 'label': 1, 'abstract': 'The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 

In [8]:
train_dataset[0]

{'pmid': '10491763',
 'relation_type': 'Association',
 'id_1': '3175',
 'id_2': 'D003924',
 'entity_a_text': 'hepatocyte nuclear factor (HNF)-6',
 'entity_b_text': 'Type II (non-insulin-dependent) diabetes mellitus',
 'perturbation': 'gold',
 'label': 2,
 'abstract': 'The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis followed by direct

In [9]:
# display how many no relation there are
print(train_df["target_relation"].value_counts())

print()

# number of entries that are not no relation
print(train_df[train_df["target_relation"] != NO_RELATION_LABEL].shape[0])


target_relation
NoRelation              3783
Association             1992
Positive_Correlation    1024
Negative_Correlation     689
Bind                      53
Cotreatment               31
Comparison                28
Drug_Interaction          11
Conversion                 3
Name: count, dtype: int64

3831


In [10]:
print(f"Relation classification: {len(RELATION_LABELS)} classes")
print("label2id:", label2id)

Relation classification: 9 classes
label2id: {'Positive_Correlation': 0, 'Negative_Correlation': 1, 'Association': 2, 'Comparison': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Bind': 6, 'Conversion': 7, 'NoRelation': 8}


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [12]:
def preprocess_function(examples, key: str = "prompt"):
    return tokenizer(
        examples[key],
        truncation=True,
        padding=True,
        max_length=512,
    )

In [13]:
def _tokenize(dataset: Dataset) -> Dataset:
    return dataset.map(
        preprocess_function,
        batched=True,
        batch_size=BATCH_SIZE,
        num_proc=NUM_PROCS,
    )

# used for training
tokenized_train = load_or_cache(
    "train", f"{DATASET_NAME}_relation_type_tokenized_{hash(str(train_dataset))}", lambda: _tokenize(train_dataset), use_cache=USE_CACHE
)

# used for validation
tokenized_valid = load_or_cache(
    "valid", f"{DATASET_NAME}_relation_type_tokenized_{hash(str(valid_dataset))}", lambda: _tokenize(valid_dataset), use_cache=USE_CACHE
)

# used for final testing
tokenized_test = load_or_cache(
    "test", f"{DATASET_NAME}_relation_type_tokenized_{hash(str(test_dataset))}", lambda: _tokenize(test_dataset), use_cache=USE_CACHE
)

Parameter 'function'=<function preprocess_function at 0x7c491c0f54e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Saving the dataset (1/1 shards): 100%|██████████| 1962/1962 [00:00<00:00, 134048.29 examples/s]


In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
# tokenized_sample = preprocess_function(train_dataset[0])
# print(tokenized_sample)

In [16]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")


def compute_metrics(eval_pred):
    """Compute accuracy and globally configured F1 for 9-way relation-type classification."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        **accuracy_metric.compute(predictions=predictions, references=labels),
        **precision_metric.compute(predictions=predictions, references=labels, average="macro", zero_division=0),
        **recall_metric.compute(predictions=predictions, references=labels, average="macro", zero_division=0),

        "f1_micro": f1_metric.compute(predictions=predictions, references=labels, average="micro")["f1"],
        "f1_macro": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(RELATION_LABELS),
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8681.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: NeuML/pubmedbert-base-embeddings
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
if MODE != "cpu":
    model = model.to(MODE)
print(f"Moving model to {MODE}")


# Train on Train.PubTator; validate on Dev.PubTator during training (Test.PubTator is held out).
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Moving model to cuda


In [19]:
# torch.cuda.empty_cache()

In [20]:
# TRAIN

history = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
1,No log,1.007080,0.618770,0.321566,0.300018,0.618770,0.293711,0.602935
2,1.129451,0.864014,0.651549,0.334842,0.347438,0.651549,0.339096,0.654522
3,0.750828,1.023583,0.684329,0.662221,0.376275,0.684329,0.420454,0.668094
4,0.470631,1.146359,0.662775,0.416373,0.479538,0.662775,0.430700,0.670006
5,0.324928,1.211632,0.691064,0.521221,0.386715,0.691064,0.409934,0.686407
6,0.232554,1.425888,0.688370,0.393100,0.413135,0.688370,0.398935,0.684501
7,0.157984,1.564792,0.691962,0.482282,0.471639,0.691962,0.468446,0.692489
8,0.124605,1.715188,0.700045,0.456932,0.482524,0.700045,0.456876,0.698293
9,0.086092,1.801933,0.694207,0.477157,0.484507,0.694207,0.467394,0.693718
10,0.059606,1.882425,0.695106,0.476389,0.511023,0.695106,0.478789,0.695836


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]


In [21]:
# Dev split: model selection / early stopping during training.
# print("=== Dev (validation) ===")
# eval_results = trainer.evaluate(tokenized_valid)
# print(eval_results)

# predictions = trainer.predict(tokenized_valid)
# pred_labels = np.argmax(predictions.predictions, axis=1)
# true_labels = predictions.label_ids

# print(
#     classification_report(
#         true_labels,
#         pred_labels,
#         labels=list(range(len(RELATION_LABELS))),
#         target_names=RELATION_LABELS,
#         digits=3,
#         zero_division=0,
#     )
# )

# Test split: held-out final evaluation (run once after training).
print("\n=== Test (held-out) ===")
test_eval_results = trainer.evaluate(tokenized_test)
print(test_eval_results)

test_predictions = trainer.predict(tokenized_test)
test_pred_labels = np.argmax(test_predictions.predictions, axis=1)
test_true_labels = test_predictions.label_ids

print(
    classification_report(
        test_true_labels,
        test_pred_labels,
        labels=list(range(len(RELATION_LABELS))),
        target_names=RELATION_LABELS,
        digits=3,
        zero_division=0,
    )
)


=== Test (held-out) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
0.059606,1.708039,10,0.694699,0.578019,0.568671,0.694699,0.566136,0.691651


{'eval_loss': 1.7080390453338623, 'eval_accuracy': 0.6946992864424058, 'eval_precision': 0.5780185418440111, 'eval_recall': 0.5686708208329475, 'eval_f1_micro': 0.6946992864424058, 'eval_f1_macro': 0.5661355000685492, 'eval_f1_weighted': 0.6916510059893698}


                      precision    recall  f1-score   support

Positive_Correlation      0.534     0.707     0.608       276
Negative_Correlation      0.643     0.675     0.659       160
         Association      0.632     0.487     0.550       526
          Comparison      0.571     0.667     0.615         6
         Cotreatment      0.889     0.571     0.696        14
    Drug_Interaction      1.000     1.000     1.000         2
                Bind      0.143     0.200     0.167         5
          Conversion      0.000     0.000     0.000         1
          NoRelation      0.790     0.812     0.801       972

            accuracy                          0.695      1962
           macro avg      0.578     0.569     0.566      1962
        weighted avg      0.698     0.695     0.692      1962



In [22]:
# append the latest evaluation scores to a JSONL registry for later comparison

RESULTS_FILE = Path("results.jsonl")


def save_result(model: str | AutoModelForSequenceClassification, metrics: dict[str, float] | None = None) -> None:
    """Append one evaluation run to ``RESULTS_FILE``, pulling the rest from the global env.

    Args:
        model: Model identifier, or a loaded model whose ``name_or_path`` is used.
        metrics: Metrics dict from ``Trainer.evaluate``; defaults to the global ``eval_results``.
    """
    metrics = eval_results if metrics is None else metrics
    model_name = model if isinstance(model, str) or isinstance(model, Path) else model.name_or_path
    record = {
        "model_name": str(model_name),
        "model": str(model_name),
        "dataset": DATASET_NAME,
        "f1_average": F1_AVERAGE,
        "metrics": {key.removeprefix("eval_"): value for key, value in metrics.items()},
    }
    with RESULTS_FILE.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, indent=4) + "\n")
    print(f"Appended result for {model} to {RESULTS_FILE}")

save_result(OUT_DIR, test_eval_results)

Appended result for relations-bert-2026-07-01-20-51-41_BioRedPerturbated_micro to results.jsonl


In [23]:
model.save_pretrained(f"{OUT_DIR}_dump")  # TODO

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


In [ ]:
OUT_DIR

In [24]:
# Load a trained checkpoint (must be 9-class relation-type model, not the old binary one).
CHECKPOINT_DIR = Path(OUT_DIR) / "checkpoint-1260"

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

if MODE != "cpu":
    model = model.to(MODE)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


print("\n=== Test (held-out) ===")
test_eval_results = trainer.evaluate(tokenized_test)
print(test_eval_results)

test_predictions = trainer.predict(tokenized_test)
test_pred_labels = np.argmax(test_predictions.predictions, axis=1)
test_true_labels = test_predictions.label_ids

print(
    classification_report(
        test_true_labels,
        test_pred_labels,
        labels=list(range(len(RELATION_LABELS))),
        target_names=RELATION_LABELS,
        digits=3,
        zero_division=0,
    )
)

OSError: relations-bert-2026-07-01-20-51-41_BioRedPerturbated_micro/checkpoint-1260 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [26]:
# call evaluate on checkpoint


def evaluate_checkpoint(
    checkpoint_dir: Path | str,
    eval_dataset: Dataset = tokenized_valid,
    split_name: str = "test",
    args: TrainingArguments = training_args,
    mode: str = MODE,
    labels: list[str] = RELATION_LABELS,
) -> dict[str, float]:
    """Load a fine-tuned checkpoint and evaluate it on ``eval_dataset``.

    Loads the model and tokenizer from ``checkpoint_dir``, runs ``Trainer.evaluate``
    for the configured metrics and prints a per-class ``classification_report``.

    Args:
        checkpoint_dir: Path to a 9-class relation-type checkpoint directory.
        eval_dataset: Tokenized dataset to evaluate on (defaults to the dev split).
        split_name: Human-readable split label for printed reports.
        args: Training arguments reused for the evaluation ``Trainer``.
        mode: Device to place the model on (``"cpu"``, ``"cuda"`` or ``"mps"``).
        labels: Ordered class names used for the classification report.

    Returns:
        The metrics dictionary returned by ``Trainer.evaluate``.
    """
    checkpoint_dir = Path(checkpoint_dir)

    checkpoint_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    checkpoint_tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    checkpoint_collator = DataCollatorWithPadding(tokenizer=checkpoint_tokenizer)

    if mode != "cpu":
        checkpoint_model = checkpoint_model.to(mode)

    checkpoint_trainer = Trainer(
        model=checkpoint_model,
        args=args,
        eval_dataset=eval_dataset,
        data_collator=checkpoint_collator,
        compute_metrics=compute_metrics,
    )

    print(f"=== {split_name} ===")
    eval_results = checkpoint_trainer.evaluate(eval_dataset)
    print(eval_results)

    predictions = checkpoint_trainer.predict(eval_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids

    print(
        classification_report(
            true_labels,
            pred_labels,
            labels=list(range(len(labels))),
            target_names=labels,
            digits=3,
            zero_division=0,
        )
    )

    return eval_results


#checkpoint_dir = Path("relations-bert-2026-06-11-14-35-34_BioRed_micro") / "checkpoint-5220"
#results = evaluate_checkpoint(checkpoint_dir)



In [25]:
# iterate over all checkpoints, find the best one based on f1_micro on test


def find_checkpoints(run_dir: Path | str) -> list[Path]:
    """Return all ``checkpoint-*`` directories in ``run_dir`` sorted by global step."""
    run_dir = Path(run_dir)
    return sorted(
        (p for p in run_dir.glob("checkpoint-*") if p.is_dir()),
        key=lambda p: int(p.name.split("-")[-1]),
    )


def score_checkpoint(
    checkpoint_dir: Path | str,
    eval_dataset: Dataset = tokenized_test,
    args: TrainingArguments = training_args,
    mode: str = MODE,
) -> dict[str, float]:
    """Evaluate a single checkpoint and return its metrics without printing a report.

    Args:
        checkpoint_dir: Path to a 9-class relation-type checkpoint directory.
        eval_dataset: Tokenized dataset to score on (defaults to the test split).
        args: Training arguments reused for the evaluation ``Trainer``.
        mode: Device to place the model on (``"cpu"``, ``"cuda"`` or ``"mps"``).

    Returns:
        The metrics dictionary returned by ``Trainer.evaluate``.
    """
    checkpoint_dir = Path(checkpoint_dir)

    checkpoint_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    checkpoint_tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    checkpoint_collator = DataCollatorWithPadding(tokenizer=checkpoint_tokenizer)

    if mode != "cpu":
        checkpoint_model = checkpoint_model.to(mode)

    checkpoint_trainer = Trainer(
        model=checkpoint_model,
        args=args,
        eval_dataset=eval_dataset,
        data_collator=checkpoint_collator,
        compute_metrics=compute_metrics,
    )

    return checkpoint_trainer.evaluate(eval_dataset)


RUN_DIR = Path("relations-bert-2026-07-01-18-38-41_BioRed_micro")
SELECTION_METRIC = f"eval_f1_{F1_AVERAGE}"

checkpoints = find_checkpoints(RUN_DIR)
if not checkpoints:
    raise FileNotFoundError(f"No checkpoint-* directories found under {RUN_DIR}")

checkpoint_results: dict[Path, dict[str, float]] = {}
for checkpoint in checkpoints:
    results = score_checkpoint(checkpoint)
    checkpoint_results[checkpoint] = results
    print(
        f"{checkpoint.name}: "
        f"f1_micro={results['eval_f1_micro']:.4f} "
        f"f1_macro={results['eval_f1_macro']:.4f} "
        f"f1_weighted={results['eval_f1_weighted']:.4f}"
    )

best_checkpoint = max(checkpoint_results, key=lambda c: checkpoint_results[c][SELECTION_METRIC])
print()
print()
print()
print(f"\nBest checkpoint: {best_checkpoint.name} ({SELECTION_METRIC}={checkpoint_results[best_checkpoint][SELECTION_METRIC]:.4f})\n")

test_results = evaluate_checkpoint(best_checkpoint, eval_dataset=tokenized_test, split_name="test")



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5131.79it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
No log,0.990112,0,0.596840,0.231585,0.224680,0.596840,0.222756,0.593419


checkpoint-522: f1_micro=0.5968 f1_macro=0.2228 f1_weighted=0.5934


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5790.41it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
No log,0.957741,0,0.629460,0.256482,0.270393,0.629460,0.261085,0.631532


...

checkpoint-4698: f1_micro=0.6453 f1_macro=0.5197 f1_weighted=0.6497


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6413.16it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
No log,2.036830,0,0.645770,0.564597,0.533687,0.645770,0.533166,0.651202


checkpoint-5220: f1_micro=0.6458 f1_macro=0.5332 f1_weighted=0.6512




Best checkpoint: checkpoint-2610 (eval_f1_micro=0.6595)



NameError: name 'evaluate_checkpoint' is not defined

In [27]:
test_results = evaluate_checkpoint(best_checkpoint, eval_dataset=tokenized_test, split_name="test")



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9885.27it/s]


=== test ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Micro,F1 Macro,F1 Weighted
No log,1.243144,0,0.659531,0.652674,0.501716,0.659531,0.533894,0.665504


{'eval_loss': 1.243144154548645, 'eval_accuracy': 0.6595310907237513, 'eval_precision': 0.6526737842983206, 'eval_recall': 0.5017164195308502, 'eval_f1_micro': 0.6595310907237513, 'eval_f1_macro': 0.533894470647464, 'eval_f1_weighted': 0.6655039239329488}


                      precision    recall  f1-score   support

Positive_Correlation      0.523     0.663     0.585       276
Negative_Correlation      0.532     0.681     0.597       160
         Association      0.565     0.625     0.594       526
          Comparison      0.800     0.667     0.727         6
         Cotreatment      0.636     0.500     0.560        14
    Drug_Interaction      1.000     0.500     0.667         2
                Bind      1.000     0.200     0.333         5
          Conversion      0.000     0.000     0.000         1
          NoRelation      0.818     0.679     0.742       972

            accuracy                          0.660      1962
           macro avg      0.653     0.502     0.534      1962
        weighted avg      0.684     0.660     0.666      1962



In [28]:
save_result(best_checkpoint, test_results)

Appended result for relations-bert-2026-07-01-18-38-41_BioRed_micro/checkpoint-2610 to results.jsonl
